In [69]:
# Mahjong CONSTS
from mahjong.constants import EAST, SOUTH, WEST, NORTH, HAKU, HATSU, CHUN #former 4 are .WINDS, and all 7 are .HONOR_INDICES
from mahjong.constants import FIVE_RED_MAN, FIVE_RED_PIN, FIVE_RED_SOU # Red Dora number cards > .AKA_DORA_LIST
from mahjong.constants import DISPLAY_WINDS # Str display of the winds

# Mahjong methods/classes
from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig, OptionalRules
from mahjong.meld import Meld
from mahjong.tile import TilesConverter

# imports
import random
import re
from typing import List, Optional, Tuple, Union
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import copy

In [70]:
TOTAL_CARD_NUM = 136 # 4 * [9pin + 9sozu + 9manzu + 3sanngennpai + 4jihai]

In [71]:
# UTILITY FUNCTIONS

class MahjongConverter:
    """
    Utility class for Mahjong tile data conversions.
    All methods are static to serve as a functional engine for wrapper classes.
    """

    @staticmethod
    def to_34_array(tiles_136: List[int]) -> List[int]:
        tiles_34 = [0] * 34
        for t in tiles_136:
            tiles_34[t // 4] += 1
        return tiles_34

    @staticmethod
    def from_34_to_136(tiles_34: List[int]) -> List[int]:
        tiles_136 = []
        for tile_type, count in enumerate(tiles_34):
            base_id = tile_type * 4
            for i in range(min(count, 4)):
                tiles_136.append(base_id + i)
        return tiles_136

    @staticmethod
    def to_str(tiles_136: List[int]) -> str:
        if not tiles_136: return ""
        t34 = MahjongConverter.to_34_array(tiles_136)
        sections = [(0, 9, "m"), (9, 18, "p"), (18, 27, "s"), (27, 34, "z")]
        res = ""
        for start, end, suffix in sections:
            group = "".join(str(i - start + 1) * t34[i] for i in range(start, end) if t34[i] > 0)
            if group: res += f"{group}{suffix}"
        return res

    @staticmethod
    def to_136(hand_str: str) -> List[int]:
        t34 = [0] * 34
        suites = re.findall(r"(\d+)([mpsz])", hand_str)
        mapping = {"m": 0, "p": 9, "s": 18, "z": 27}
        for digits, suffix in suites:
            for d in digits:
                idx = (int(d) - 1) + mapping[suffix]
                if t34[idx] < 4: t34[idx] += 1
        return MahjongConverter.from_34_to_136(t34)

class MahjongBase:
    """Provides shared comparison and arithmetic logic."""
    def to_34(self) -> List[int]:
        raise NotImplementedError

    def to_ids(self) -> List[int]:
        raise NotImplementedError

    def __eq__(self, other: object) -> bool:
        if isinstance(other, (str, list, int)):
            if isinstance(other, str): other = MSPZ(other)
            elif isinstance(other, list): other = Hand136(other)
            elif isinstance(other, int): other = Hand136([other])
        
        if hasattr(other, 'to_34'):
            return self.to_34() == other.to_34()
        return False

    def __add__(self, other: Union['MahjongBase', str, list, int]) -> 'Hand136':
        """Allows combining hands using the + operator."""
        # Convert 'self' IDs
        self_ids = self.to_ids()
        
        # Convert 'other' to IDs
        if isinstance(other, MahjongBase):
            other_ids = other.to_ids()
        elif isinstance(other, str):
            other_ids = MahjongConverter.to_136(other)
        elif isinstance(other, list):
            other_ids = other
        elif isinstance(other, int):
            other_ids = [other]
        else:
            raise TypeError(f"Unsupported operand type for +: 'MahjongBase' and '{type(other).__name__}'")
            
        return Hand136(self_ids + other_ids)

    def __radd__(self, other: Union[str, list, int]) -> 'Hand136':
        """Handles cases where raw types are on the left side of +."""
        return self.__add__(other)

class Hand136(MahjongBase):
    def __init__(self, ids: Union[int, List[int]]):
        # Flatten list if needed and sort for consistency
        self.ids = sorted(ids) if isinstance(ids, list) else [ids]

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(self.ids)

    def to_ids(self) -> List[int]:
        return self.ids

    def to_mspz(self) -> 'MSPZ':
        return MSPZ(MahjongConverter.to_str(self.ids))
    
    def remove(self, id):
        self.ids.remove(id)
    
    def __repr__(self):
        return f"Hand136({self.ids})"
    
    def __len__(self) -> int:
        """Allows calling len(hand_obj)"""
        return len(self.ids)

    def __iter__(self):
        """Allows for tile in hand_obj: ..."""
        return iter(self.ids)
    
    def __getitem__(self, position):
        """
        Allows for calling via index
        """
        return self.ids[position]

class MSPZ(MahjongBase):
    def __init__(self, notation: str):
        self.notation = notation

    def to_34(self) -> List[int]:
        return MahjongConverter.to_34_array(MahjongConverter.to_136(self.notation))

    def to_ids(self) -> List[int]:
        return MahjongConverter.to_136(self.notation)

    def to_136(self) -> Hand136:
        return Hand136(self.to_ids())

    def __repr__(self):
        return f"MSPZ('{self.notation}')"

def make_random_hand() -> Hand136:
    """Random generation of tiles of size 13 and 1 drawn tile

    Returns
    -------
    List[Hand136]
        Hand136 len==13, Hand136 len==1
    """
    all_tiles_i = list(range(TOTAL_CARD_NUM))
    random.shuffle(all_tiles_i)
    hand_tiles = sorted(all_tiles_i[:13])
    drawn_tile = all_tiles_i[13]
    return Hand136(hand_tiles), Hand136(drawn_tile)


In [72]:
# GAME SIMULTION CLASS
class MahjongPlayer:
    def __init__(self, name: str, initial_score: int = 35000):
        self.name = name
        self.score = initial_score
        self.hand: Optional[Hand136] = None  # Current hidden tiles
        self.discards: List[Hand136] = []    # History of 136-IDs discarded
        self.melds: List[str] = []           # Calls like 'pon', 'chi', 'kan'
        self.is_riichi = False
        self.is_dealer = False

    def decide_discard(self) -> Hand136:
        discard_tile = Hand136(random.choice(self.hand))
        return discard_tile

    def discard_tile(self, tile: Hand136):
        # Implementation to remove from self.hand and add to self.discards
        self.hand.remove(tile)
        self.discards.append(tile)
        return tile

    def __repr__(self):
        dealer_mark = " (Dealer)" if self.is_dealer else ""
        return f"[{self.name}]{dealer_mark} Score: {self.score}"
    
class MahjongTable:
    BA = ["East", "South", "West", "North"]
    def __init__(self, rules: dict):
        self.rules = rules or {
            "starting_score": 25000,
            "ending_score": 30000,
            "num_players": 4,
            "player_names": [],
            "game_len": 2 # Tonpuu=1, Hanchan=2
        }
        if not rules:
            self.rules["player_names"] = [f"P{i}" for i in range(self.rules["num_players"])]
        self.players = List[MahjongPlayer]
        self.wall: List[Hand136] = []
        self.dead_wall: List[int]
        self.turn_index = 0  # 0: East, 1: South, 2: West, 3: North
        self.round_number = 0 # E1, E2...
        # etc.
        self.dora_indicators: List[int]
        self.honba = 0 # Counter for dealer repeats, Renchan
        self.log = []
                
    @property
    def tiles_remaining(self) -> int:
        """The core metric for Game Theory: How many turns are left in the round?"""
        return len(self.wall)

    def is_game_over(self) -> bool:
        """Game ends when round_number reaches rule's game length"""
        return self.round_number == self.rules["game_len"]

    def is_round_over(self) -> bool:
        """Hand ends when no tiles remain to be drawn."""
        return self.tiles_remaining == 0

    def setup_round(self):
        """Initializes a new hand: Shuffles wall, deals tiles, sets Dora."""
        # Reinitialize field
        self.wall = list([Hand136(n) for n in list(range(136))])
        random.shuffle(self.wall)
        self.dead_wall = [self.wall.pop() for _ in range(14)]
        self.dora_indicators = [self.dead_wall[0]] # First Dora

        # Reinitialize all players
        self.players = [MahjongPlayer(name, self.rules["starting_score"]) for name in self.rules["player_names"]]
        dealer_idx = (self.round_number - 1) % 4
        for i, p in enumerate(self.players):
            p.hand = Hand136([self.wall.pop().ids[0] for _ in range(13)])
            p.is_dealer = (i == dealer_idx)
            
        self._log_state("SETUP ROUND")

    def get_current_player(self) -> MahjongPlayer:
        return self.players[self.turn_index]

    def next_turn(self):
        self.turn_index = (self.turn_index + 1) % self.rules["num_players"]

    def play_round(self):
        """A simplified loop representing one full hand (Kyoku)."""
        while not self.is_round_over():
            current_player = self.get_current_player()
            current_player: MahjongPlayer
            
            # DRAW
            new_tile = self.wall.pop()
            self._log_state("DRAW", new_tile.to_mspz())
            current_player.hand += new_tile

            # DISCARD (AI Decision Point)
            discarded_tile = current_player.decide_discard()
            current_player.discard_tile(discarded_tile)
            self._log_state("DISCARD", discarded_tile.to_mspz())
            
            self.next_turn()
        self.round_number += 1
    
    def play_game(self):
        """Loop representing a full game"""
        while not self.is_game_over():
            self.setup_round()
            self.play_round()
            self._log_state("ROUND OVER")
        self._log_state("GAME OVER")

    def _log_state(self, action_type: str, tile_id: str = None):
        """Captures the current state of the table for post-game review."""
        snapshot = {
            "round_number": self.round_number,
            "turn_index": self.turn_index,
            "tiles_remaining": self.tiles_remaining,
            "dora": copy.deepcopy(self.dora_indicators),
            "action": action_type,
            "action_tile": tile_id,
            "players": [
                {
                    "name": p.name,
                    "score": p.score,
                    "hand": p.hand.to_ids() if p.hand else [],
                    "discards": list(p.discards),
                    "is_dealer": p.is_dealer
                } for p in self.players
            ]
        }
        self.log.append(snapshot)

In [73]:
# VISUALIZE SIMULATION

class MahjongReplay:
    def __init__(self, history):
        self.history = history
        self.current_idx = 0
        self.output = widgets.Output()
        # Find all indices where a new round starts
        self.round_start_indices = self._get_round_indices()

    def _get_round_indices(self):
        indices = [i for i,n in enumerate(self.history) if n["action"] == "SETUP ROUND"]
        return indices

    def render(self):
        """Standardized render call to update the output area."""
        if not self.history:
            with self.output:
                print("Error: History is empty. Run play_game() first.")
            return

        state = self.history[self.current_idx]
        with self.output:
            clear_output(wait=True)
            
            # Use a slightly simpler HTML structure to ensure cross-platform compatibility
            html = f"""
            <div style="background-color: #222; color: #eee; padding: 15px; border-radius: 8px; font-family: monospace;">
                <div style="border-bottom: 1px solid #444; padding-bottom: 5px; margin-bottom: 10px;">
                    <b>{MahjongTable.BA[state['round_number']]} {state['round_number']}</b> | 
                    Wall: {state['tiles_remaining']} | Step: {self.current_idx}
                </div>
                <div style="color: #0f0; margin-bottom: 10px;">Action: {state['action']} {state.get('action_tile', '')}</div>
            """
            
            for i, p in enumerate(state['players']):
                active = "background: #333; border: 1px solid #0f0;" if i == state['turn_index'] else ""
                hand = MSPZ(MahjongConverter.to_str(p['hand'])).notation
                
                html += f"""
                <div style="padding: 5px; margin: 2px; {active}">
                    {p['name']} ({p['score']}): {hand}
                </div>
                """
            html += "</div>"
            display(HTML(html))

    # --- Navigation Logic ---
    def move_next_turn(self, _):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def move_prev_turn(self, _):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def move_next_round(self, _):
        for idx in self.round_start_indices:
            if idx > self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def move_prev_round(self, _):
        for idx in reversed(self.round_start_indices):
            if idx < self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def show(self):
        """Build and display the entire UI."""
        # Create buttons
        b1 = widgets.Button(description="PREV R", layout=widgets.Layout(width='80px'))
        b2 = widgets.Button(description="PREV T", layout=widgets.Layout(width='80px'))
        b3 = widgets.Button(description="NEXT T", layout=widgets.Layout(width='80px'))
        b4 = widgets.Button(description="NEXT R", layout=widgets.Layout(width='80px'))

        # Navigation logic (direct functions)
        def p_r(_): 
            for idx in reversed(self.round_start_indices):
                if idx < self.current_idx:
                    self.current_idx = idx; break
            self.render()
        
        def p_t(_): 
            self.current_idx = max(0, self.current_idx - 1); self.render()
            
        def n_t(_): 
            self.current_idx = min(len(self.history)-1, self.current_idx + 1); self.render()
            
        def n_r(_):
            for idx in self.round_start_indices:
                if idx > self.current_idx:
                    self.current_idx = idx; break
            self.render()

        b1.on_click(p_r); b2.on_click(p_t); b3.on_click(n_t); b4.on_click(n_r)

        # Trigger initial render
        self.render()
        
        # Display everything directly
        ui = widgets.VBox([widgets.HBox([b1, b2, b3, b4]), self.output])
        display(ui)

In [74]:
rules = {
    "starting_score": 25000,
    "ending_score": 30000,
    "num_players": 4,
    "player_names": ["sakana", "ono", "lynn", "yagata"],
    "game_len": 1 # Tonpuu=1, Hanchan=2
}
table = MahjongTable(rules=None)
table.play_game()
game_log = table.log

replay = MahjongReplay(game_log)
replay.show()